In [103]:
import pandas as pd
from langchain_community.embeddings import OllamaEmbeddings
import os
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv
from pinecone_text.sparse import BM25Encoder
from langchain_community.retrievers import PineconeHybridSearchRetriever
import pickle

In [49]:
load_dotenv()
pine_api_key = os.getenv("PINECONE_API_KEY")

In [42]:
OLLAMA_BASE_URL = "http://localhost:11434"

embeddings = OllamaEmbeddings(
    model="qwen3-embedding:8b",
    base_url=OLLAMA_BASE_URL,
)


In [29]:
test_vec = embeddings.embed_query("testing")
dim = len(test_vec)
print("Embedding dimension:", dim)


Embedding dimension: 4096


# Ebn al Kathir tafseer

In [105]:
df = pd.read_csv("tafseer_ebnalkatheer_final.csv")
df

,surah_no,ayah_start,ayah_end,ayah_and_tafseer
0,1,1,1,ayah: In the Name of Allah-the Most Compassion...
1,1,2,2,ayah: All praise is for Allah-Lord of all worl...
2,1,3,3,"ayah: the Most Compassionate, Most Merciful,\n..."
3,1,4,4,ayah: Master of the Day of Judgment.\ntafseer:...
4,1,5,5,ayah: You ˹alone˺ we worship and You ˹alone˺ w...
...,...,...,...,...
1891,110,1,3,ayah: When Allah's ˹ultimate˺ help comes and t...
1892,111,1,5,"ayah: May the hands of Abu Lahab perish, and h..."
1893,112,1,4,"ayah: Say, ˹O Prophet,˺ ""He is Allah-One ˹and ..."
1894,113,1,5,"ayah: Say, ˹O Prophet,˺ ""I seek refuge in the ..."


In [106]:
INDEX_NAME = "quran-tafseer-ebn"

pc = Pinecone(api_key=pine_api_key)
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=dim,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
index = pc.Index(INDEX_NAME)

In [107]:
texts = df["ayah_and_tafseer"].astype(str).tolist()

metadatas = [
    {
        "surah_no": int(row.surah_no),
        "ayah_start": int(row.ayah_start),
        "ayah_end": int(row.ayah_end),
    }
    for row in df.itertuples(index=False)
]

In [108]:
ids = [
    f"{int(row.surah_no)}:{int(row.ayah_start)}-{int(row.ayah_end)}"
    for row in df.itertuples(index=False)
]

In [109]:
bm25 = BM25Encoder()
bm25.fit(texts)

  0%|          | 0/1896 [00:00<?, ?it/s]

In [110]:
with open("bm25_quran_ebn.pkl", "wb") as f:
    pickle.dump(bm25, f)

In [84]:
NAMESPACE = "quran_ebn"


retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25,
    index=index,
    namespace=NAMESPACE,
    top_k=10,
    alpha=0.7,   # start here
)

In [85]:
retriever.add_texts(
    texts=texts,
    metadatas=metadatas,
    ids=ids,
    namespace="quran_jal",
)

  0%|          | 0/60 [00:00<?, ?it/s]

In [86]:
stats = index.describe_index_stats(namespace="quran_ebn")
print(stats)

{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'quran_jal': {'vector_count': 1896}},
 'total_vector_count': 1896,
 'vector_type': 'dense'}


# Jalalyin tafseer

In [87]:
df_jal = pd.read_csv("rag_dataset.csv")
df_jal

,surah_name,surah_number,ayah_no,ayah_ar,ayah_and_tafseer
0,Al-Fatihah,1,1,بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ,ayah: in the name of allah-the most compassion...
1,Al-Fatihah,1,2,ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَٰلَمِينَ,ayah: all praise is for allah-lord of all worl...
2,Al-Fatihah,1,3,ٱلرَّحْمَٰنِ ٱلرَّحِيمِ,"ayah: the most compassionate, most merciful,\n..."
3,Al-Fatihah,1,4,مَٰلِكِ يَوْمِ ٱلدِّينِ,ayah: master of the day of judgment.\ntafseer:...
4,Al-Fatihah,1,5,إِيَّاكَ نَعْبُدُ وَإِيَّاكَ نَسْتَعِينُ,ayah: you ˹alone˺ we worship and you ˹alone˺ w...
...,...,...,...,...,...
6231,An-Nas,114,2,مَلِكِ ٱلنَّاسِ,"ayah: the master of humankind,\ntafseer: the k..."
6232,An-Nas,114,3,إِلَٰهِ ٱلنَّاسِ,"ayah: the god of humankind,\ntafseer: the god ..."
6233,An-Nas,114,4,مِن شَرِّ ٱلْوَسْوَاسِ ٱلْخَنَّاسِ,ayah: from the evil of the lurking whisperer-\...
6234,An-Nas,114,5,ٱلَّذِى يُوَسْوِسُ فِى صُدُورِ ٱلنَّاسِ,ayah: who whispers into the hearts of humankin...


In [88]:
new_df = pd.DataFrame({
    "surah_no": df_jal["surah_number"],
    "ayah_start": df_jal["ayah_no"],
    "ayah_end": df_jal["ayah_no"],
    "ayah_and_tafseer": df_jal["ayah_and_tafseer"]
})
df = new_df
df

,surah_no,ayah_start,ayah_end,ayah_and_tafseer
0,1,1,1,ayah: in the name of allah-the most compassion...
1,1,2,2,ayah: all praise is for allah-lord of all worl...
2,1,3,3,"ayah: the most compassionate, most merciful,\n..."
3,1,4,4,ayah: master of the day of judgment.\ntafseer:...
4,1,5,5,ayah: you ˹alone˺ we worship and you ˹alone˺ w...
...,...,...,...,...
6231,114,2,2,"ayah: the master of humankind,\ntafseer: the k..."
6232,114,3,3,"ayah: the god of humankind,\ntafseer: the god ..."
6233,114,4,4,ayah: from the evil of the lurking whisperer-\...
6234,114,5,5,ayah: who whispers into the hearts of humankin...


In [89]:
INDEX_NAME = "quran-tafseer-jal"

pc = Pinecone(api_key=pine_api_key)
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=dim,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
index = pc.Index(INDEX_NAME)

In [90]:
texts = df["ayah_and_tafseer"].astype(str).tolist()

metadatas = [
    {
        "surah_no": int(row.surah_no),
        "ayah_start": int(row.ayah_start),
        "ayah_end": int(row.ayah_end),
    }
    for row in df.itertuples(index=False)
]

In [91]:
ids = [
    f"{int(row.surah_no)}:{int(row.ayah_start)}-{int(row.ayah_end)}"
    for row in df.itertuples(index=False)
]

In [92]:
bm25 = BM25Encoder()
bm25.fit(texts)

  0%|          | 0/6236 [00:00<?, ?it/s]

In [104]:
with open("bm25_quran_jal.pkl", "wb") as f:
    pickle.dump(bm25, f)

In [93]:
NAMESPACE = "quran_jal"


retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25,
    index=index,
    namespace=NAMESPACE,
    top_k=10,
    alpha=0.7,   # start here
)

In [95]:
retriever.add_texts(
    texts=texts,
    metadatas=metadatas,
    ids=ids,
    namespace="quran_jal",
)

  0%|          | 0/195 [00:00<?, ?it/s]

In [97]:
stats = index.describe_index_stats(namespace="quran_jal")
print(stats)

{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'quran_jal': {'vector_count': 6236}},
 'total_vector_count': 6236,
 'vector_type': 'dense'}


In [102]:
query = "What does joseph say to his father?"

retriever.top_k = 10
retriever.alpha = 0.7
docs = retriever.invoke("What does joseph dream?")

for i, d in enumerate(docs, 1):
    print(f"\n--- Result {i} ---")
    print(d.metadata)
    print(d.page_content[:600])



--- Result 1 ---
{'ayah_end': 46.0, 'ayah_start': 46.0, 'surah_no': 12.0, 'score': 0.6313}
ayah: ˹he said,˺ "joseph, o man of truth! interpret for us ˹the dream of˺ seven fat cows eaten up by seven skinny ones; and seven green ears of grain and ˹seven˺ others dry, so that i may return to the people and let them know."
tafseer: 'o joseph o truthful one one given to truth give us your opinion concerning seven fat kine that are devoured by seven lean ones and concerning seven green ears of corn and seven others dry that i may return to the people that is to the king and his courtiers so that they might know' its interpretation.

--- Result 2 ---
{'ayah_end': 36.0, 'ayah_start': 36.0, 'surah_no': 12.0, 'score': 0.595370412}
ayah: and two other servants went to jail with joseph. one of them said, "i dreamt i was pressing wine." the other said, "i dreamt i was carrying ˹some˺ bread on my head, from which birds were eating." ˹then both said,˺ "tell us their interpretation, for we surely see 